In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [1]:
# =========================
# 1. Imports
# =========================
import os, json
import numpy as np
import pandas as pd
from scipy.sparse import hstack

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from xgboost import XGBClassifier

# =========================
# 2. LOAD JSONL DATA
# =========================
base_path = "/kaggle/input/datasets/nikunjnawal009/xgboost-primevul26"

def load_jsonl(file):
    X, y =[],[]
    if not os.path.exists(file):
        raise FileNotFoundError(f"❌ File not found: {file}")
        
    with open(file, "r", encoding="utf-8") as f:
        for line in f:
            obj = json.loads(line)

            code = (
                obj.get("func_before") or
                obj.get("code") or
                obj.get("func") or
                ""
            )

            label = obj.get("target", obj.get("label", 0))

            if code.strip():
                X.append(code)
                y.append(int(label))

    return X, y

print("Loading PrimeVul datasets...")
X_train, y_train = load_jsonl(os.path.join(base_path, "primevul_train_paired.jsonl"))
X_val,   y_val   = load_jsonl(os.path.join(base_path, "primevul_valid_paired.jsonl"))
X_test,  y_test  = load_jsonl(os.path.join(base_path, "primevul_test_paired.jsonl"))

# 🔥 UPGRADE: Do NOT merge val_data into train_data! Strict isolation.
print(f"Train size: {len(X_train)} | Val size: {len(X_val)} | Test size: {len(X_test)}")

# =========================
# 3. DUAL FEATURE ENGINEERING (Word + Char)
# =========================
print("\n⚙️ Vectorizing Data (Dual Word+Char N-Grams)...")

# 3A. Word-Level Vectorizer (Syntax & Keywords)
vec_word = TfidfVectorizer(
    max_features=15000,
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.90,
    # 🔥 UPGRADE: Captures punctuation/operators like ==, ->, {
    token_pattern=r'[a-zA-Z0-9_]+|[^\w\s]', 
    dtype=np.float32
)

# 3B. Char-Level Vectorizer (Motifs, Typos, & Structural Patterns)
vec_char = TfidfVectorizer(
    max_features=15000,
    analyzer='char_wb',
    ngram_range=(3, 5), 
    min_df=3,
    max_df=0.90,
    dtype=np.float32
)

# Fit on Training data ONLY
print("Fitting vectorizers to Training data...")
X_tr_w = vec_word.fit_transform(X_train)
X_tr_c = vec_char.fit_transform(X_train)

X_train_vec = hstack([X_tr_w, X_tr_c]).tocsr()

# Transform Val and Test Sets
X_val_vec  = hstack([vec_word.transform(X_val), vec_char.transform(X_val)]).tocsr()
X_test_vec = hstack([vec_word.transform(X_test), vec_char.transform(X_test)]).tocsr()

print(f"Total Combined Features: {X_train_vec.shape[1]}")

# =========================
# 4. HANDLE CLASS IMBALANCE
# =========================
# Safely calculate class ratio for scale_pos_weight
pos_weight = (len(y_train) - sum(y_train)) / sum(y_train)
print(f"Scale Pos Weight calculated as: {pos_weight:.2f}")

# =========================
# 5. XGBOOST MODEL
# =========================
print("\n🚀 Training XGBoost Classifier (This may take a moment)...")

model = XGBClassifier(
    n_estimators=400,          # Added more trees since we lowered learning rate
    learning_rate=0.05,        # Slower learning prevents overfitting on dense text features
    max_depth=7,               # Slightly deeper trees to capture "If A + B + C" code logic
    subsample=0.8,
    colsample_bytree=0.8,      # Prevents trees from relying entirely on a single obvious keyword
    scale_pos_weight=pos_weight,
    eval_metric="logloss",
    random_state=42,
    n_jobs=-1
)

model.fit(X_train_vec, y_train)

# =========================
# 6. MACRO-F1 THRESHOLD TUNING (ON VAL SET)
# =========================
val_probs = model.predict_proba(X_val_vec)[:, 1]

best_macro_f1 = 0
best_t = 0.5

print("\n🔍 Threshold tuning strictly on Validation Set:")

# Fast mathematical loop across thresholds
for t in np.arange(0.20, 0.82, 0.02):
    preds = (val_probs > t).astype(int)
    
    tn, fp, fn, tp = confusion_matrix(y_val, preds, labels=[0, 1]).ravel()
    
    recall_0 = tn / (tn + fp) if (tn + fp) > 0 else 0
    precision_0 = tn / (tn + fn) if (tn + fn) > 0 else 0
    f1_0 = 2 * (precision_0 * recall_0) / (precision_0 + recall_0) if (precision_0 + recall_0) > 0 else 0

    recall_1 = tp / (tp + fn) if (tp + fn) > 0 else 0
    precision_1 = tp / (tp + fp) if (tp + fp) > 0 else 0
    f1_1 = 2 * (precision_1 * recall_1) / (precision_1 + recall_1) if (precision_1 + recall_1) > 0 else 0

    macro_f1 = (f1_0 + f1_1) / 2

    if round(t * 100) % 10 == 0:
        print(f"t={t:.2f} → Macro_F1={macro_f1:.4f} | R0={recall_0:.2f}, R1={recall_1:.2f}")

    if macro_f1 > best_macro_f1:
        best_macro_f1 = macro_f1
        best_t = t

print(f"\nBest threshold found: {best_t:.2f}")

# =========================
# 7. FINAL EVALUATION (ON TEST SET)
# =========================
print("\nEvaluating untouched Test Set...")
test_probs = model.predict_proba(X_test_vec)[:, 1]
final_preds = (test_probs > best_t).astype(int)

acc = accuracy_score(y_test, final_preds)
report = classification_report(y_test, final_preds, output_dict=True)

print("\n✅ Accuracy:", acc)
print("\n📊 Classification Report:\n", classification_report(y_test, final_preds))

# =========================
# 8. SAVE RESULTS
# =========================
label_key = "1" if "1" in report else[k for k in report.keys() if str(k).startswith("1")][0]

results = {
    "Model": "XGBoost_Dual_TFIDF",
    "Dataset": "PrimeVul",
    "Accuracy": acc,
    "Precision_vuln": report[label_key]['precision'],
    "Recall_vuln": report[label_key]['recall'],
    "F1_vuln": report[label_key]['f1-score'],
    "Macro_F1": report['macro avg']['f1-score'],
    "Best_threshold": best_t
}

pd.DataFrame([results]).to_csv("/kaggle/working/xgb_primevul_results.csv", index=False)

print("\n✅ Results saved in /kaggle/working/")

Loading PrimeVul datasets...
Train size: 7578 | Val size: 960 | Test size: 870

⚙️ Vectorizing Data (Dual Word+Char N-Grams)...
Fitting vectorizers to Training data...
Total Combined Features: 30000
Scale Pos Weight calculated as: 1.00

🚀 Training XGBoost Classifier (This may take a moment)...

🔍 Threshold tuning strictly on Validation Set:
t=0.20 → Macro_F1=0.3324 | R0=0.00, R1=1.00
t=0.30 → Macro_F1=0.3781 | R0=0.05, R1=0.97
t=0.40 → Macro_F1=0.4890 | R0=0.23, R1=0.85
t=0.50 → Macro_F1=0.5310 | R0=0.51, R1=0.56
t=0.60 → Macro_F1=0.4685 | R0=0.87, R1=0.19
t=0.70 → Macro_F1=0.3561 | R0=1.00, R1=0.02
t=0.80 → Macro_F1=0.3333 | R0=1.00, R1=0.00

Best threshold found: 0.52

Evaluating untouched Test Set...

✅ Accuracy: 0.5275862068965518

📊 Classification Report:
               precision    recall  f1-score   support

           0       0.52      0.66      0.58       435
           1       0.54      0.40      0.46       435

    accuracy                           0.53       870
   macro a